# SENTINEL — Insider Threat Detection System
## Academic Analysis Notebook
**CERT r4.2 Dataset | Isolation Forest + Autoencoder Ensemble**

This notebook covers:
1. Exploratory Data Analysis (EDA)
2. Feature Distribution Analysis
3. Model Performance Evaluation
4. ROC Curve & AUC
5. Feature Importance
6. Insider vs Normal Comparison
7. Risk Score Distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Dark detective theme
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#111820',
    'axes.edgecolor':   '#2a3444',
    'axes.labelcolor':  '#8b98ab',
    'text.color':       '#e8edf5',
    'xtick.color':      '#8b98ab',
    'ytick.color':      '#8b98ab',
    'grid.color':       '#1a2332',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
    'axes.titlecolor':  '#e8edf5',
    'figure.dpi':       120,
})

ACCENT  = '#e63946'
COLORS  = {'CRITICAL':'#ff2d55','HIGH':'#ff6b35','MEDIUM':'#ffd60a','LOW':'#30d158','NORMAL':'#3a8fb7'}
INSIDER = '#ff2d55'
NORMAL  = '#3a8fb7'

print('Libraries loaded. Ready for analysis.')

## 1. Load All Data

In [ ]:
# Raw logs
logon  = pd.read_csv('../data/raw/logon.csv',  low_memory=False)
files  = pd.read_csv('../data/raw/file.csv',   low_memory=False)
device = pd.read_csv('../data/raw/device.csv', low_memory=False)
email  = pd.read_csv('../data/raw/email.csv',  low_memory=False)
ldap   = pd.read_csv('../data/raw/LDAP.csv')
gt     = pd.read_csv('../data/raw/ground_truth.csv')

# Processed
features    = pd.read_parquet('../data/processed/daily_behavior.parquet')
scored      = pd.read_parquet('../data/processed/scored_behaviors.parquet')
user_risk   = pd.read_parquet('../data/processed/user_risk_scores.parquet')
alerts      = pd.read_parquet('../data/processed/alerts.parquet')
X_all       = np.load('../data/models/X_all.npy')
meta        = pd.read_parquet('../data/models/X_meta.parquet')

print(f'Raw data loaded:')
print(f'  Logon events:  {len(logon):>8,}')
print(f'  File events:   {len(files):>8,}')
print(f'  Device events: {len(device):>8,}')
print(f'  Email events:  {len(email):>8,}')
print(f'  Users (LDAP):  {len(ldap):>8,}')
print(f'  Insider flags: {gt["is_insider"].sum():>8,}')
print(f'\nProcessed:')
print(f'  Feature matrix:  {features.shape}')
print(f'  Scored rows:     {scored.shape}')
print(f'  User risk scores:{user_risk.shape}')
print(f'  Alerts:          {len(alerts)}')

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('DATASET OVERVIEW', fontsize=14, color='#e8edf5', y=0.98)

# 1. Dept distribution
ax = axes[0,0]
dept_counts = ldap['department'].value_counts()
bars = ax.barh(dept_counts.index, dept_counts.values, color=ACCENT, alpha=0.8)
ax.set_title('Users per Department')
ax.set_xlabel('Count')
for bar, val in zip(bars, dept_counts.values):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=9)

# 2. Login hours distribution
ax = axes[0,1]
logon['hour'] = pd.to_datetime(logon['date'], errors='coerce').dt.hour
normal_users  = set(gt[~gt['is_insider']]['user_id'])
insider_users = set(gt[gt['is_insider']]['user_id'])
ax.hist(logon[logon['user'].isin(normal_users)]['hour'].dropna(),
        bins=24, alpha=0.7, color=NORMAL, label='Normal', density=True)
ax.hist(logon[logon['user'].isin(insider_users)]['hour'].dropna(),
        bins=24, alpha=0.7, color=INSIDER, label='Insider', density=True)
ax.set_title('Login Hour Distribution')
ax.set_xlabel('Hour of Day')
ax.axvspan(0, 8, alpha=0.08, color='red')
ax.axvspan(18, 24, alpha=0.08, color='red')
ax.legend(fontsize=9)

# 3. Email external rate
ax = axes[0,2]
email['is_external'] = ~email['to'].str.contains('dtaa.com', na=False)
email_by_user = email.groupby('user')['is_external'].mean() * 100
n_scores = email_by_user[email_by_user.index.isin(normal_users)]
i_scores = email_by_user[email_by_user.index.isin(insider_users)]
ax.hist(n_scores, bins=20, alpha=0.7, color=NORMAL,  label='Normal', density=True)
ax.hist(i_scores, bins=10, alpha=0.7, color=INSIDER, label='Insider', density=True)
ax.set_title('External Email Rate (%)')
ax.legend(fontsize=9)

# 4. USB usage
ax = axes[1,0]
usb_by_user = device.groupby('user').size()
n_usb = usb_by_user[usb_by_user.index.isin(normal_users)]
i_usb = usb_by_user[usb_by_user.index.isin(insider_users)]
ax.hist(n_usb, bins=20, alpha=0.7, color=NORMAL,  label='Normal', density=True)
ax.hist(i_usb, bins=10, alpha=0.7, color=INSIDER, label='Insider', density=True)
ax.set_title('USB Event Count per User')
ax.set_xlabel('USB Events')
ax.legend(fontsize=9)

# 5. Risk tier distribution
ax = axes[1,1]
tier_order = ['CRITICAL','HIGH','MEDIUM','LOW','NORMAL']
tier_counts = user_risk['risk_tier'].value_counts().reindex(tier_order, fill_value=0)
tier_colors = [COLORS[t] for t in tier_order]
bars = ax.bar(tier_order, tier_counts.values, color=tier_colors, alpha=0.85, edgecolor='none')
ax.set_title('Risk Tier Distribution')
ax.set_ylabel('Users')
for bar, val in zip(bars, tier_counts.values):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, val+0.3, str(val), ha='center', fontsize=10)

# 6. Risk score histogram
ax = axes[1,2]
ax.hist(user_risk['max_risk_score'], bins=25, color=ACCENT, alpha=0.8, edgecolor='none')
for thresh, label, col in [(85,'HIGH','#ff6b35'),(70,'MED','#ffd60a'),(40,'LOW','#30d158')]:
    ax.axvline(thresh, color=col, linestyle='--', alpha=0.7, linewidth=1.2)
    ax.text(thresh+1, ax.get_ylim()[1]*0.8, label, color=col, fontsize=8)
ax.set_title('Max Risk Score Distribution')
ax.set_xlabel('Max Risk Score')
ax.set_ylabel('Users')

plt.tight_layout()
plt.savefig('../reports/eda_overview.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('EDA overview saved → reports/eda_overview.png')

## 3. Feature Correlation Heatmap

In [ ]:
import json
with open('../data/models/feature_names.json') as f:
    feat_names = json.load(f)

feat_df = pd.DataFrame(X_all, columns=feat_names)
corr = feat_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
fig.patch.set_facecolor('#0d1117')
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=0.8, vmin=-0.8,
            center=0, square=True, linewidths=0.3, ax=ax,
            cbar_kws={"shrink": 0.6},
            annot=False, xticklabels=True, yticklabels=True)
ax.set_title('FEATURE CORRELATION MATRIX', pad=15, fontsize=12)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('../reports/feature_correlation.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4. Insider vs Normal — Feature Comparison

In [ ]:
# Merge features with ground truth
feat_gt = features.merge(gt[['user_id','is_insider','threat_type']], 
                          left_on='user', right_on='user_id', how='left')
feat_gt['is_insider'] = feat_gt['is_insider'].fillna(False)

key_features = ['files_accessed','usb_count','emails_external',
                'after_hours_ratio','sensitive_files','email_attachments',
                'files_accessed_deviation','usb_count_deviation']
key_features = [f for f in key_features if f in feat_gt.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('INSIDER vs NORMAL — KEY FEATURE DISTRIBUTIONS', fontsize=13, color='#e8edf5')

for ax, feat in zip(axes.flat, key_features):
    normal_vals  = feat_gt[~feat_gt['is_insider']][feat].dropna()
    insider_vals = feat_gt[feat_gt['is_insider']][feat].dropna()
    
    p95 = normal_vals.quantile(0.99)
    normal_vals  = normal_vals[normal_vals <= p95 * 2]
    insider_vals = insider_vals[insider_vals <= p95 * 2]
    
    ax.hist(normal_vals,  bins=30, alpha=0.6, color=NORMAL,  density=True, label='Normal')
    ax.hist(insider_vals, bins=15, alpha=0.6, color=INSIDER, density=True, label='Insider')
    ax.set_title(feat.replace('_',' ').title(), fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('Value', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/feature_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Model Evaluation — ROC Curve & AUC

In [ ]:
from src.models.isolation_forest_model import IsolationForestDetector
from src.models.autoencoder_model import AutoencoderDetector
from src.models.scorer import ScoringEngine

# Load models
ifd = IsolationForestDetector(); ifd.load()
ae  = AutoencoderDetector();  ae.load()
eng = ScoringEngine()

# Get per-user avg scores
if_scores  = ifd.predict_scores(X_all)
ae_scores  = ae.predict_scores(X_all)
ens_scores = eng.ensemble_score(if_scores, ae_scores)

meta_scores = meta.copy()
meta_scores['if_score']  = if_scores
meta_scores['ae_score']  = ae_scores
meta_scores['ens_score'] = ens_scores

user_if  = meta_scores.groupby('user')['if_score'].mean().reset_index()
user_ae  = meta_scores.groupby('user')['ae_score'].mean().reset_index()
user_ens = meta_scores.groupby('user')['ens_score'].mean().reset_index()

def get_roc(user_scores_df, score_col):
    merged = user_scores_df.merge(gt[['user_id','is_insider']], 
                                   left_on='user', right_on='user_id', how='inner')
    y_true  = merged['is_insider'].astype(int).values
    y_score = merged[score_col].values
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc

fpr_if,  tpr_if,  auc_if  = get_roc(user_if,  'if_score')
fpr_ae,  tpr_ae,  auc_ae  = get_roc(user_ae,  'ae_score')
fpr_ens, tpr_ens, auc_ens = get_roc(user_ens, 'ens_score')

print(f'Isolation Forest AUC-ROC:  {auc_if:.4f}')
print(f'Autoencoder AUC-ROC:       {auc_ae:.4f}')
print(f'Ensemble AUC-ROC:          {auc_ens:.4f}')

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#0d1117')
ax.plot([0,1],[0,1], 'k--', alpha=0.3, label='Random')
ax.plot(fpr_if,  tpr_if,  color='#3a8fb7', lw=2, label=f'Isolation Forest (AUC={auc_if:.3f})')
ax.plot(fpr_ae,  tpr_ae,  color='#ffd60a', lw=2, label=f'Autoencoder (AUC={auc_ae:.3f})')
ax.plot(fpr_ens, tpr_ens, color=ACCENT,    lw=2.5, label=f'Ensemble (AUC={auc_ens:.3f})')
ax.fill_between(fpr_ens, tpr_ens, alpha=0.08, color=ACCENT)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC CURVE — MODEL COMPARISON', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/roc_curve.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 6. Detection Rate — Top-K Analysis

In [ ]:
insider_ids  = set(gt[gt['is_insider']]['user_id'])
n_insiders   = len(insider_ids)
n_users      = len(user_risk)

user_ens_merged = user_ens.merge(gt[['user_id','is_insider']], 
                                  left_on='user', right_on='user_id', how='inner')
user_ens_sorted = user_ens_merged.sort_values('ens_score', ascending=False)

ks = range(1, n_users + 1)
precisions = []
recalls    = []
for k in ks:
    top_k    = user_ens_sorted.head(k)
    detected = top_k['is_insider'].sum()
    precisions.append(detected / k)
    recalls.append(detected / n_insiders)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('TOP-K DETECTION PERFORMANCE', fontsize=12, color='#e8edf5')

ax1.plot(list(ks), precisions, color=ACCENT, lw=2)
ax1.axvline(n_insiders, color='#ffd60a', linestyle='--', alpha=0.7)
ax1.text(n_insiders+0.5, 0.5, f'k={n_insiders}\n(all insiders)', color='#ffd60a', fontsize=9)
ax1.set_xlabel('k (top users examined)')
ax1.set_ylabel('Precision@k')
ax1.set_title('Precision@K')
ax1.grid(True, alpha=0.3)

ax2.plot(list(ks), recalls, color='#30d158', lw=2)
ax2.axhline(1.0, color='#ffd60a', linestyle='--', alpha=0.4)
ax2.set_xlabel('k (top users examined)')
ax2.set_ylabel('Recall@k')
ax2.set_title('Recall@K (Insider Coverage)')
ax2.grid(True, alpha=0.3)

# Print detection summary
print(f'\nDetection Summary:')
print(f'  Total users:      {n_users}')
print(f'  Known insiders:   {n_insiders}')
for k in [5, 10, 15, 20, n_insiders]:
    if k <= n_users:
        top_k    = user_ens_sorted.head(k)
        detected = top_k['is_insider'].sum()
        print(f'  Top-{k:2d} precision: {detected/k:.1%}  recall: {detected/n_insiders:.1%}  ({detected}/{n_insiders} insiders found)')

plt.tight_layout()
plt.savefig('../reports/detection_rate.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 7. Risk Score Timeline — Insider Activity Spikes

In [ ]:
scored['day'] = pd.to_datetime(scored['day'])
insiders_list = list(insider_ids)
colors_cycle  = [INSIDER, '#ff6b35', '#ffd60a', '#c77dff', '#00b4d8', '#f72585', '#4cc9f0', '#80ffdb']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('RISK SCORE TIMELINE — INSIDER vs POPULATION', fontsize=12, color='#e8edf5')

# Population average
pop_daily = scored.groupby('day')['risk_score'].agg(['mean','max']).reset_index()
ax1.fill_between(pop_daily['day'], pop_daily['mean'], alpha=0.15, color=NORMAL)
ax1.plot(pop_daily['day'], pop_daily['mean'], color=NORMAL, lw=1.5, label='Population avg', alpha=0.8)
ax1.plot(pop_daily['day'], pop_daily['max'],  color='#ff6b35', lw=1.2, linestyle='--', label='Population max', alpha=0.6)
ax1.axhline(85, color=ACCENT, linestyle=':', alpha=0.5)
ax1.axhline(70, color='#ffd60a', linestyle=':', alpha=0.4)
ax1.set_ylabel('Risk Score')
ax1.set_title('POPULATION RISK SCORE OVER TIME')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 105)
ax1.grid(True, alpha=0.2)

# Individual insiders
for i, uid in enumerate(insiders_list[:6]):
    user_daily = scored[scored['user'] == uid].sort_values('day')
    threat_row = gt[gt['user_id'] == uid].iloc[0] if uid in gt['user_id'].values else None
    col = colors_cycle[i % len(colors_cycle)]
    label = f'{uid} ({threat_row["threat_type"] if threat_row is not None else "unknown"})'
    ax2.plot(user_daily['day'], user_daily['risk_score'], color=col, lw=1.5, label=label, alpha=0.9)
    # Mark trigger date
    if threat_row is not None and pd.notna(threat_row['trigger_date']):
        td = pd.to_datetime(threat_row['trigger_date'])
        ax2.axvline(td, color=col, linestyle='--', alpha=0.4, linewidth=0.8)

ax2.axhline(85, color=ACCENT, linestyle=':', alpha=0.5, label='HIGH threshold (85)')
ax2.set_ylabel('Risk Score')
ax2.set_xlabel('Date')
ax2.set_title('INDIVIDUAL INSIDER RISK SCORES (dashed lines = trigger dates)')
ax2.legend(fontsize=8, loc='upper left')
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/risk_timeline.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 8. Feature Importance (Permutation-Based)

In [ ]:
# Permutation importance: shuffle each feature and measure score degradation
import json
with open('../data/models/feature_names.json') as f:
    feat_names = json.load(f)

# Get baseline user scores
ifd  = IsolationForestDetector(); ifd.load()
base = ifd.predict_scores(X_all)

importances = []
rng_local   = np.random.default_rng(42)

for i, fname in enumerate(feat_names):
    X_perm       = X_all.copy()
    X_perm[:, i] = rng_local.permutation(X_perm[:, i])
    perm_scores  = ifd.predict_scores(X_perm)
    delta        = np.mean(np.abs(perm_scores - base))
    importances.append((fname, delta))

importances.sort(key=lambda x: x[1], reverse=True)
names, deltas = zip(*importances)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1117')
colors = [ACCENT if d > np.mean(deltas) else '#3a8fb7' for d in deltas]
bars = ax.barh(names[::-1], [d for d in deltas[::-1]], color=colors[::-1], alpha=0.85)
ax.set_title('FEATURE IMPORTANCE (Permutation Method)', fontsize=12)
ax.set_xlabel('Mean Absolute Score Change When Permuted')
ax.axvline(np.mean(deltas), color='#ffd60a', linestyle='--', alpha=0.6, label='Mean importance')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\nTop 10 most important features:')
for rank, (name, score) in enumerate(importances[:10], 1):
    print(f'  {rank:2d}. {name:<40s}  Δ={score:.4f}')

## 9. Threat Archetype Comparison

In [ ]:
archetypes = gt[gt['is_insider']].merge(
    user_risk[['user','max_risk_score','avg_risk_score','risk_tier','trend']],
    left_on='user_id', right_on='user', how='left'
)
archetypes = archetypes.merge(
    ldap[['user_id','name','department','role']],
    on='user_id', how='left'
)

print('INSIDER THREAT DOSSIER')
print('='*90)
for _, row in archetypes.iterrows():
    print(f"  {row['user_id']} | {row['name']:25s} | {row['department']:12s} | "
          f"{row['threat_type']:22s} | "
          f"Trigger: {row['trigger_date']} | "
          f"Max: {row.get('max_risk_score',0):5.1f} | Tier: {row.get('risk_tier','?')}")

# Bar chart by archetype
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0d1117')
arch_scores = archetypes.groupby('threat_type')['max_risk_score'].agg(['mean','max','min'])
x = np.arange(len(arch_scores))
w = 0.25
ax.bar(x-w, arch_scores['max'],  w, color=ACCENT,    label='Max score',  alpha=0.85)
ax.bar(x,   arch_scores['mean'], w, color='#ff6b35', label='Mean score', alpha=0.85)
ax.bar(x+w, arch_scores['min'],  w, color='#ffd60a', label='Min score',  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([t.replace('_',' ').title() for t in arch_scores.index], rotation=10)
ax.set_ylabel('Risk Score')
ax.set_title('RISK SCORES BY THREAT ARCHETYPE')
ax.legend(fontsize=9)
ax.axhline(85, color='#ff2d55', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('../reports/archetype_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 10. Summary Report

In [ ]:
print('='*70)
print('  SENTINEL — SYSTEM EVALUATION SUMMARY')
print('='*70)
print(f'  Dataset: CERT-like r4.2 (synthetic)')
print(f'  Users:             {len(ldap):>6}')
print(f'  Insider threats:   {gt["is_insider"].sum():>6}')
print(f'  Total events:      {len(logon)+len(files)+len(device)+len(email):>6,}')
print(f'  Features:          {X_all.shape[1]:>6}')
print(f'  Feature rows:      {X_all.shape[0]:>6,}')
print()
print(f'  MODELS:')
print(f'    Isolation Forest AUC-ROC:  {auc_if:.4f}')
print(f'    Autoencoder AUC-ROC:       {auc_ae:.4f}')
print(f'    Ensemble AUC-ROC:          {auc_ens:.4f}')
print()
print(f'  ALERTS GENERATED:  {len(alerts)}')
if len(alerts) > 0:
    for tier in ['CRITICAL','HIGH','MEDIUM','LOW']:
        count = (alerts['risk_tier'] == tier).sum()
        if count > 0:
            print(f'    {tier:10s}: {count}')
print()
print(f'  RISK TIER DISTRIBUTION:')
for tier, count in user_risk['risk_tier'].value_counts().items():
    pct = count / len(user_risk) * 100
    bar = '█' * int(pct/2)
    print(f'    {tier:10s}: {count:3d} ({pct:4.1f}%) {bar}')
print('='*70)
print()
print('  Reports saved to reports/')
print('  - eda_overview.png')
print('  - feature_correlation.png')
print('  - feature_comparison.png')
print('  - roc_curve.png')
print('  - detection_rate.png')
print('  - risk_timeline.png')
print('  - feature_importance.png')
print('  - archetype_comparison.png')